In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt

from scipy.special import ellipk, ellipe

from ipywidgets import (
    FloatSlider,
    HTML,
    HTMLMath,
    VBox,
    HBox,
    Layout
)

from IPython.display import display

# ============================================================
# COMPLETE ELLIPTIC INTEGRALS
#
# K(k), K'(k), E(k), E'(k)
#
# IMPORTANT:
# scipy.special uses the parameter
#
#     m = k^2
#
# while the book uses the elliptic modulus k.
# ============================================================

plt.ioff()

# ============================================================
# CLASSIC JUPYTER + JUPYTERLAB / NOTEBOOK 7 / BINDER
#
# - no forced scrollbars
# - no trimming
# - no resize triangle
# ============================================================

display(HTML("""
<style>

.container {
    width: 98% !important;
    max-width: none !important;
}

.output_area,
.output_subarea {
    max-width: none !important;
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
}

.output_scroll {
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
    box-shadow: none !important;
}

.jp-Cell-outputWrapper,
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    max-width: none !important;
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
}

.widget-output,
.jupyter-widgets-output-area,
.widget-box {
    max-width: none !important;
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
}

.jupyter-matplotlib,
.jupyter-matplotlib-figure {
    overflow: visible !important;
    resize: none !important;
}

.jupyter-matplotlib::-webkit-resizer,
.jupyter-matplotlib-figure::-webkit-resizer {
    display: none !important;
}

.ell-title {
    font-family: Arial, sans-serif;
    font-size: 20px;
    font-weight: bold;
    color: #6f3fa0;
}

.ell-label {
    font-family: Arial, sans-serif;
    font-size: 14px;
    font-weight: bold;
}

.ell-value {
    font-family: Arial, sans-serif;
    font-size: 14px;
    font-weight: bold;
    color: #0b3d91;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    width:1180px;
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.48;
    margin-bottom:10px;
">

<div class="ell-title" style="margin-bottom:8px;">
Complete Elliptic Integrals
</div>

<div style="margin-bottom:5px;">
The complete elliptic integral of the first kind is denoted by K(k),
while its complementary form is K′(k)=K(k′), where
k′=(1−k²)<sup>1/2</sup>.
</div>

<div style="margin-bottom:5px;">
The complete elliptic integral of the second kind is denoted by E(k),
with complementary form E′(k)=E(k′).
</div>

<div>
<b>This notebook:</b> displays K(k), K′(k), E(k) and E′(k) as functions
of the elliptic modulus k and shows their values for the selected k.
Internally, SciPy uses the parameter m=k².
</div>

</div>
""")

# ============================================================
# MATHEMATICAL DEFINITIONS
# ============================================================

K_definition = HTMLMath(
    value=(
        r'\('
        r'K(k)'
        r'='
        r'\displaystyle\int_0^{\pi/2}'
        r'\frac{d\theta}'
        r'{\sqrt{1-k^2\sin^2\theta}}'
        r'\)'
    )
)

E_definition = HTMLMath(
    value=(
        r'\('
        r'E(k)'
        r'='
        r'\displaystyle\int_0^{\pi/2}'
        r'\sqrt{1-k^2\sin^2\theta}'
        r'\,d\theta'
        r'\)'
    )
)

complement_definition = HTMLMath(
    value=(
        r'\('
        r'k^{\prime}'
        r'='
        r'\sqrt{1-k^2}'
        r'\)'
    )
)

definition_panel = HBox(
    [
        K_definition,
        E_definition,
        complement_definition
    ],
    layout=Layout(
        width='1150px',
        gap='25px',
        align_items='center',
        overflow='visible'
    )
)

# ============================================================
# SLIDER
# ============================================================

slider_style = {
    'description_width': '0px'
}

k_slider = FloatSlider(
    min=0.01,
    max=0.99,
    step=0.01,
    value=0.50,
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=Layout(
        width='280px'
    )
)

k_label = HTML(
    '<div class="ell-label">Elliptic modulus k:</div>',
    layout=Layout(
        width='145px',
        min_width='145px'
    )
)

k_value = HTML(
    '<div class="ell-value">0.50</div>',
    layout=Layout(
        width='65px',
        min_width='65px',
        margin='0px 0px 0px 6px'
    )
)

k_row = HBox(
    [
        k_label,
        k_slider,
        k_value
    ],
    layout=Layout(
        width='510px',
        height='42px',
        align_items='center'
    )
)

# ============================================================
# CURRENT VALUES PANEL
# ============================================================

current_values_math = HTMLMath()

current_values_panel = VBox(
    [
        HTML("""
        <div class="ell-title" style="margin-bottom:7px;">
            Current Values
        </div>
        """),

        current_values_math
    ],
    layout=Layout(
        width='610px',
        padding='10px 14px',
        border='1px solid #d2c2df',
        overflow='visible'
    )
)

# ============================================================
# PARAMETERS PANEL
# ============================================================

parameters_panel = VBox(
    [
        HTML("""
        <div class="ell-title" style="margin-bottom:7px;">
            Parameter
        </div>
        """),

        k_row
    ],
    layout=Layout(
        width='520px',
        padding='10px 14px',
        border='1px solid #d2c2df',
        overflow='visible'
    )
)

# ============================================================
# TOP ROW
# ============================================================

top_row = HBox(
    [
        parameters_panel,
        current_values_panel
    ],
    layout=Layout(
        width='1160px',
        gap='15px',
        align_items='stretch',
        overflow='visible'
    )
)

# ============================================================
# k GRID
#
# We avoid the exact endpoints 0 and 1 because K or K'
# diverges there.
# ============================================================

k_axis = np.linspace(
    0.01,
    0.99,
    1500
)

m_axis = (
    k_axis**2
)

kp_axis = np.sqrt(
    1.0
    -
    k_axis**2
)

mp_axis = (
    kp_axis**2
)

# ============================================================
# COMPLETE ELLIPTIC INTEGRALS
#
# SciPy convention:
#
# ellipk(m)
# ellipe(m)
#
# with m = k^2
# ============================================================

K_axis = ellipk(
    m_axis
)

Kp_axis = ellipk(
    mp_axis
)

E_axis = ellipe(
    m_axis
)

Ep_axis = ellipe(
    mp_axis
)

# ============================================================
# FIGURE 1
# K(k) AND K'(k)
# ============================================================

fig_K, ax_K = plt.subplots(
    figsize=(5.7, 4.4)
)

fig_K.canvas.header_visible = False
fig_K.canvas.footer_visible = False
fig_K.canvas.toolbar_visible = False

fig_K.canvas.layout = Layout(
    width='570px',
    height='440px',
    overflow='visible'
)

ax_K.set_title(
    'Complete Elliptic Integrals K(k) and K′(k)',
    fontsize=13,
    fontweight='bold',
    color='#6f3fa0'
)

ax_K.set_xlabel(
    'Elliptic modulus k',
    fontsize=10
)

ax_K.set_ylabel(
    'Integral value',
    fontsize=10
)

ax_K.set_xlim(
    0.0,
    1.0
)

# ------------------------------------------------------------
# Fixed y-axis
#
# For k in [0.01, 0.99] these limits show both curves clearly.
# ------------------------------------------------------------

ax_K.set_ylim(
    1.3,
    6.3
)

ax_K.grid(
    True,
    linestyle=':',
    alpha=0.40
)

# ============================================================
# CURVES CREATED ONCE
# ============================================================

line_K, = ax_K.plot(
    k_axis,
    K_axis,
    linewidth=2.0,
    label='K(k)'
)

line_Kp, = ax_K.plot(
    k_axis,
    Kp_axis,
    linewidth=2.0,
    linestyle='--',
    label='K′(k)'
)

# ============================================================
# CURRENT VALUE MARKERS
# ============================================================

point_K, = ax_K.plot(
    [],
    [],
    linestyle='None',
    marker='o',
    markersize=7
)

point_Kp, = ax_K.plot(
    [],
    [],
    linestyle='None',
    marker='o',
    markersize=7
)

vertical_K = ax_K.axvline(
    k_slider.value,
    linestyle=':',
    linewidth=1.2
)

# ============================================================
# LEGEND BELOW FIGURE
# ============================================================

ax_K.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.16),
    ncol=2,
    fontsize=9,
    frameon=True
)

fig_K.subplots_adjust(
    left=0.13,
    right=0.97,
    top=0.89,
    bottom=0.25
)

# ============================================================
# FIGURE 2
# E(k) AND E'(k)
# ============================================================

fig_E, ax_E = plt.subplots(
    figsize=(5.7, 4.4)
)

fig_E.canvas.header_visible = False
fig_E.canvas.footer_visible = False
fig_E.canvas.toolbar_visible = False

fig_E.canvas.layout = Layout(
    width='570px',
    height='440px',
    overflow='visible'
)

ax_E.set_title(
    'Complete Elliptic Integrals E(k) and E′(k)',
    fontsize=13,
    fontweight='bold',
    color='#0b3d91'
)

ax_E.set_xlabel(
    'Elliptic modulus k',
    fontsize=10
)

ax_E.set_ylabel(
    'Integral value',
    fontsize=10
)

ax_E.set_xlim(
    0.0,
    1.0
)

ax_E.set_ylim(
    0.95,
    1.65
)

ax_E.grid(
    True,
    linestyle=':',
    alpha=0.40
)

# ============================================================
# CURVES CREATED ONCE
# ============================================================

line_E, = ax_E.plot(
    k_axis,
    E_axis,
    linewidth=2.0,
    label='E(k)'
)

line_Ep, = ax_E.plot(
    k_axis,
    Ep_axis,
    linewidth=2.0,
    linestyle='--',
    label='E′(k)'
)

# ============================================================
# CURRENT VALUE MARKERS
# ============================================================

point_E, = ax_E.plot(
    [],
    [],
    linestyle='None',
    marker='o',
    markersize=7
)

point_Ep, = ax_E.plot(
    [],
    [],
    linestyle='None',
    marker='o',
    markersize=7
)

vertical_E = ax_E.axvline(
    k_slider.value,
    linestyle=':',
    linewidth=1.2
)

# ============================================================
# LEGEND BELOW FIGURE
# ============================================================

ax_E.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.16),
    ncol=2,
    fontsize=9,
    frameon=True
)

fig_E.subplots_adjust(
    left=0.13,
    right=0.97,
    top=0.89,
    bottom=0.25
)

# ============================================================
# FIGURES ROW
# ============================================================

figures_row = HBox(
    [
        fig_K.canvas,
        fig_E.canvas
    ],
    layout=Layout(
        width='1160px',
        gap='15px',
        align_items='flex-start',
        overflow='visible'
    )
)

# ============================================================
# LIMIT INFORMATION
# ============================================================

limits_panel = HTML("""
<div style="
    width:1130px;
    padding:9px 12px;
    border:1px solid #d7c7e5;
    font-family:Arial, sans-serif;
    font-size:14px;
    line-height:1.5;
    box-sizing:border-box;
">

<b style="color:#6f3fa0;">
Limiting behavior:
</b>

&nbsp;

K(0)=π/2,
K(k)→∞ as k→1,
K′(k)→∞ as k→0,
and K′(1)=π/2.

</div>
""")

# ============================================================
# UPDATE FUNCTION
#
# No clear_output()
# No figure recreation
# No axis rescaling
#
# Only marker positions and displayed values are updated.
# ============================================================

def update_notebook(change=None):

    k = (
        k_slider.value
    )

    m = (
        k**2
    )

    kp = np.sqrt(
        1.0
        -
        k**2
    )

    mp = (
        kp**2
    )

    # ========================================================
    # CURRENT INTEGRAL VALUES
    # ========================================================

    K = ellipk(
        m
    )

    Kp = ellipk(
        mp
    )

    E = ellipe(
        m
    )

    Ep = ellipe(
        mp
    )

    # ========================================================
    # SLIDER VALUE
    # ========================================================

    k_value.value = (
        f'<div class="ell-value">{k:.2f}</div>'
    )

    # ========================================================
    # DISPLAY VALUES
    #
    # No quad / qquad / qqquad spacing commands are used.
    # ========================================================

    current_values_math.value = (
        r'\('
        r'k='
        +
        f'{k:.4f}'
        +
        r',\;'
        r'k^{\prime}='
        +
        f'{kp:.4f}'
        +
        r',\;'
        r'K='
        +
        f'{K:.6f}'
        +
        r',\;'
        r'K^{\prime}='
        +
        f'{Kp:.6f}'
        +
        r',\;'
        r'E='
        +
        f'{E:.6f}'
        +
        r',\;'
        r'E^{\prime}='
        +
        f'{Ep:.6f}'
        +
        r'\)'
    )

    # ========================================================
    # UPDATE K FIGURE
    # ========================================================

    point_K.set_data(
        [k],
        [K]
    )

    point_Kp.set_data(
        [k],
        [Kp]
    )

    vertical_K.set_xdata(
        [k, k]
    )

    # ========================================================
    # UPDATE E FIGURE
    # ========================================================

    point_E.set_data(
        [k],
        [E]
    )

    point_Ep.set_data(
        [k],
        [Ep]
    )

    vertical_E.set_xdata(
        [k, k]
    )

    # ========================================================
    # REDRAW ONLY
    # ========================================================

    fig_K.canvas.draw_idle()
    fig_E.canvas.draw_idle()

# ============================================================
# CONNECT SLIDER
# ============================================================

k_slider.observe(
    update_notebook,
    names='value'
)

# ============================================================
# INITIAL UPDATE
# ============================================================

update_notebook()

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        documentation,
        definition_panel,
        top_row,
        figures_row,
        limits_panel
    ],
    layout=Layout(
        width='1180px',
        gap='10px',
        overflow='visible'
    )
)

display(
    main_layout
)